# NB34 — CFTR Panel: Literatür-Destekli Feature Engineering + Ablasyon

**TEKNOFEST Sağlıkta Yapay Zeka | Genetik Varyant Patojenite Tahmini**

## Amaç

NB20 S0c_COMBINED modeli (LOO-MCC=0.644, Boot-F1=0.863, Precision=1.0) FE olmadan çalışıyor.
Bu notebook, CFTR'ye özel 16 feature (5 grup, missingness hariç) ekleyerek LOO-MCC'yi iyileştirmeyi hedefler.

### Feature Grupları
- **Grup A** (3): FCS-tarzı frekans × konservasyon (fcs_like_9 r=-0.492)
- **Grup B** (5): EK skor birleşimleri (ek_mean_top3 r=0.476)
- **Grup D** (4): AA fizikokimyasal delta'lar (hydro_abs r=0.170)
- **Grup E** (4): AA substitüsyon skorları + sınıf geçişleri (grantham ~r=0.20)
- **Grup C** ❌: Missingness — CFTR'de anlamsız (r<0.05), çıkarıldı

### Deneyler
- **Exp 1**: Grup ablasyonu (her grubun solo katkısı)
- **Exp 2**: FE versiyon karşılaştırması (no-FE vs NB16 FE vs full CFTR FE vs optimal subset)

**Birincil metrik**: LOO-CV MCC (n=111, tüm 21 benign değerlendirmede)
**İkincil metrik**: Bootstrap %80/20 pathogenic-F1 (N=50)
**Baseline**: NB20 S0c_COMBINED LOO-MCC=0.644, Boot-F1=0.863

In [1]:
# Cell 1: Imports & Config
import os, sys, warnings, gc
from datetime import datetime
from copy import deepcopy
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay,
    average_precision_score
)
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
    PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, REPORTS_DIR
from src import columns_real as CR
import lightgbm as lgb

np.random.seed(SEED)

PANEL             = "CFTR"
HIGH_MISSING_THR  = 0.50
FINAL_BENIGN_FRAC = 0.80
N_BOOT            = 50
BOOT_SEED         = SEED
PI_TEST           = 0.20

DATA_DIR    = os.path.join(PROJECT_ROOT, "data", "real_data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v12_cftr_fe")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SEED         : {SEED}")
print(f"RESULTS_DIR  : {RESULTS_DIR}")

PROJECT_ROOT : /Users/tefe/teknofest_model/teknofest_model
SEED         : 42
RESULTS_DIR  : /Users/tefe/teknofest_model/teknofest_model/results/v12_cftr_fe


In [2]:
# Cell 2: Veri Yukleme + Sutun Temizligi
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

def load_panel(name):
    return pd.read_csv(os.path.join(DATA_DIR, CR.PANEL_INFO[name]["file"]))

master   = load_panel("MASTER")
kanser   = load_panel("KANSER")
pah      = load_panel("PAH")
cftr_raw = load_panel("CFTR")

feature_cols_all = [c for c in master.columns if c not in (ID_COL, TARGET)]

def drop_exact_duplicates(df, ref_df, name):
    ref_index = {}
    for _, row in ref_df.iterrows():
        key = (row[ID_COL],) + tuple((np.nan if pd.isna(v) else v)
                                     for v in row[feature_cols_all].values)
        ref_index[key] = row[TARGET]
    drop_idx = []
    for idx, row in df.iterrows():
        key = (row[ID_COL],) + tuple((np.nan if pd.isna(v) else v)
                                     for v in row[feature_cols_all].values)
        if key in ref_index and ref_index[key] == row[TARGET]:
            drop_idx.append(idx)
    df_clean = df.drop(index=drop_idx).reset_index(drop=True)
    print(f"{name}: {df.shape[0]} -> {df_clean.shape[0]} (drop={len(drop_idx)} birebir-ayni)")
    return df_clean

cftr         = drop_exact_duplicates(cftr_raw, master, "CFTR")
kanser_clean = drop_exact_duplicates(kanser, cftr_raw, "KANSER")
pah_clean    = drop_exact_duplicates(pah, cftr_raw, "PAH")

combined = pd.concat([master, kanser_clean, pah_clean], ignore_index=True)

print(f"\nMASTER  : {master.shape}, label: {master[TARGET].value_counts().to_dict()}")
print(f"KANSER  : {kanser_clean.shape}, label: {kanser_clean[TARGET].value_counts().to_dict()}")
print(f"PAH     : {pah_clean.shape}, label: {pah_clean[TARGET].value_counts().to_dict()}")
print(f"CFTR    : {cftr.shape}, label: {cftr[TARGET].value_counts().to_dict()}")
print(f"COMBINED: {combined.shape}, label: {combined[TARGET].value_counts().to_dict()}")

constant_cols     = CR.get_constant_cols(master[feature_cols_all])
dup_pairs         = CR.get_duplicate_col_pairs(master[feature_cols_all])
dup_drop          = sorted({b for (a, b) in dup_pairs})
drop_cols         = sorted(set(constant_cols) | set(dup_drop))
base_feature_cols = [c for c in feature_cols_all if c not in drop_cols]
CAT_LIKE          = [c for c in (CR.CAT_COLS + CR.AA_COLS) if c in base_feature_cols]
NUM_COLS_BASE     = [c for c in base_feature_cols if c not in CAT_LIKE]

print(f"\nSutun temizligi: drop {len(drop_cols)} -> {len(base_feature_cols)} feature")
print(f"  ({len(NUM_COLS_BASE)} sayisal + {len(CAT_LIKE)} kategorik)")

CFTR: 111 -> 111 (drop=0 birebir-ayni)
KANSER: 388 -> 388 (drop=0 birebir-ayni)
PAH: 372 -> 372 (drop=0 birebir-ayni)

MASTER  : (2931, 353), label: {1: 2149, 0: 782}
KANSER  : (388, 353), label: {1: 268, 0: 120}
PAH     : (372, 353), label: {1: 310, 0: 62}
CFTR    : (111, 353), label: {1: 90, 0: 21}
COMBINED: (3691, 353), label: {1: 2727, 0: 964}

Sutun temizligi: drop 63 -> 288 feature
  (281 sayisal + 7 kategorik)


In [3]:
# Cell 3: Feature Engineering -- add_fe_cftr() (16 feature, 5 grup, missingness haric)
AA_UNK      = CR.AA_UNKNOWN_TOKEN
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# === Grantham Distance Matrix ===
_GRANTHAM = {
 ('S','R'):110,('S','L'):145,('S','P'):74,('S','T'):58,('S','A'):99,('S','V'):124,
 ('S','G'):56,('S','I'):142,('S','F'):155,('S','Y'):144,('S','C'):112,('S','H'):89,
 ('S','Q'):68,('S','N'):46,('S','K'):121,('S','D'):65,('S','E'):80,('S','M'):135,('S','W'):177,
 ('R','L'):102,('R','P'):103,('R','T'):71,('R','A'):112,('R','V'):96,('R','G'):125,('R','I'):97,
 ('R','F'):97,('R','Y'):77,('R','C'):180,('R','H'):29,('R','Q'):43,('R','N'):86,('R','K'):26,
 ('R','D'):96,('R','E'):54,('R','M'):91,('R','W'):101,
 ('L','P'):98,('L','T'):92,('L','A'):96,('L','V'):32,('L','G'):138,('L','I'):5,('L','F'):22,
 ('L','Y'):36,('L','C'):198,('L','H'):99,('L','Q'):113,('L','N'):153,('L','K'):107,('L','D'):172,
 ('L','E'):138,('L','M'):15,('L','W'):61,
 ('P','T'):38,('P','A'):27,('P','V'):68,('P','G'):42,('P','I'):95,('P','F'):114,('P','Y'):110,
 ('P','C'):169,('P','H'):77,('P','Q'):76,('P','N'):91,('P','K'):103,('P','D'):108,('P','E'):93,
 ('P','M'):87,('P','W'):147,
 ('T','A'):58,('T','V'):69,('T','G'):59,('T','I'):89,('T','F'):103,('T','Y'):92,('T','C'):149,
 ('T','H'):47,('T','Q'):42,('T','N'):65,('T','K'):78,('T','D'):85,('T','E'):65,('T','M'):81,('T','W'):128,
 ('A','V'):64,('A','G'):60,('A','I'):94,('A','F'):113,('A','Y'):112,('A','C'):195,('A','H'):86,
 ('A','Q'):91,('A','N'):111,('A','K'):106,('A','D'):126,('A','E'):107,('A','M'):84,('A','W'):148,
 ('V','G'):109,('V','I'):29,('V','F'):50,('V','Y'):55,('V','C'):192,('V','H'):84,('V','Q'):96,
 ('V','N'):133,('V','K'):97,('V','D'):152,('V','E'):121,('V','M'):21,('V','W'):88,
 ('G','I'):135,('G','F'):153,('G','Y'):147,('G','C'):159,('G','H'):98,('G','Q'):87,('G','N'):80,
 ('G','K'):127,('G','D'):94,('G','E'):98,('G','M'):127,('G','W'):184,
 ('I','F'):21,('I','Y'):33,('I','C'):198,('I','H'):94,('I','Q'):109,('I','N'):149,('I','K'):102,
 ('I','D'):168,('I','E'):134,('I','M'):10,('I','W'):61,
 ('F','Y'):22,('F','C'):205,('F','H'):100,('F','Q'):116,('F','N'):158,('F','K'):102,('F','D'):177,
 ('F','E'):140,('F','M'):28,('F','W'):40,
 ('Y','C'):194,('Y','H'):83,('Y','Q'):99,('Y','N'):143,('Y','K'):85,('Y','D'):160,('Y','E'):122,
 ('Y','M'):36,('Y','W'):37,
 ('C','H'):174,('C','Q'):154,('C','N'):139,('C','K'):202,('C','D'):154,('C','E'):170,('C','M'):196,('C','W'):215,
 ('H','Q'):24,('H','N'):68,('H','K'):32,('H','D'):81,('H','E'):40,('H','M'):87,('H','W'):115,
 ('Q','N'):46,('Q','K'):53,('Q','D'):61,('Q','E'):29,('Q','M'):101,('Q','W'):130,
 ('N','K'):94,('N','D'):23,('N','E'):42,('N','M'):142,('N','W'):174,
 ('K','D'):101,('K','E'):56,('K','M'):95,('K','W'):110,
 ('D','E'):45,('D','M'):160,('D','W'):181,
 ('E','M'):126,('E','W'):152,
 ('M','W'):67,
}
def grantham(a, b):
    if a == b: return 0
    return _GRANTHAM.get((a, b)) or _GRANTHAM.get((b, a)) or -1

# === BLOSUM62 Matrix ===
_B62_RAW = """A4 R-1 N-2 D-2 C0 Q-1 E-1 G0 H-2 I-1 L-1 K-1 M-1 F-2 P-1 S1 T0 W-3 Y-2 V0
R5 N0 D-2 C-3 Q1 E0 G-2 H0 I-3 L-2 K2 M-1 F-3 P-2 S-1 T-1 W-3 Y-2 V-3
N6 D1 C-3 Q0 E0 G0 H1 I-3 L-3 K0 M-2 F-3 P-2 S1 T0 W-4 Y-2 V-3
D6 C-3 Q0 E2 G-1 H-1 I-3 L-4 K-1 M-3 F-3 P-1 S0 T-1 W-4 Y-3 V-3
C9 Q-3 E-4 G-3 H-3 I-1 L-1 K-3 M-1 F-2 P-3 S-1 T-1 W-2 Y-2 V-1
Q5 E2 G-2 H0 I-3 L-2 K1 M0 F-3 P-1 S0 T-1 W-2 Y-1 V-2
E5 G-2 H0 I-3 L-3 K1 M-2 F-3 P-1 S0 T-1 W-3 Y-2 V-2
G6 H-2 I-4 L-4 K-2 M-3 F-3 P-2 S0 T-2 W-2 Y-3 V-3
H8 I-3 L-3 K-1 M-2 F-1 P-2 S-1 T-2 W-2 Y2 V-3
I4 L2 K-3 M1 F0 P-3 S-2 T-1 W-3 Y-1 V3
L4 K-2 M2 F0 P-3 S-2 T-1 W-2 Y-1 V1
K5 M-1 F-3 P-1 S0 T-1 W-3 Y-2 V-2
M5 F0 P-2 S-1 T-1 W-1 Y-1 V1
F6 P-4 S-2 T-2 W1 Y3 V-1
P7 S-1 T-1 W-4 Y-3 V-2
S4 T1 W-3 Y-2 V-2
T5 W-2 Y-2 V0
W11 Y2 V-3
Y7 V-1
V4"""
_ORDER = list("ARNDCQEGHILKMFPSTWYV")
_B62 = {}
for _ri, _line in enumerate(_B62_RAW.strip().split("\n")):
    _toks = _line.split()
    _row_aa = _toks[0][0]
    _vals = [_toks[0][1:]] + _toks[1:]
    for _ci, _tok in enumerate(_vals):
        _col_aa = _ORDER[_ri + _ci]
        _v = int(_tok[1:] if _tok[0].isalpha() else _tok)
        _B62[(_row_aa, _col_aa)] = _v; _B62[(_col_aa, _row_aa)] = _v
def blosum62(a, b): return _B62.get((a, b), 0)

# === Physicochemical Lookup Tables ===
_HYDRO = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,
           'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,
           'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}
_VOLUME = {'A':88.6,'R':173.4,'N':114.1,'D':111.1,'C':108.5,'Q':143.8,'E':138.4,'G':60.1,
            'H':153.2,'I':166.7,'L':166.7,'K':168.6,'M':162.9,'F':189.9,'P':112.7,'S':89.0,
            'T':116.1,'W':227.8,'Y':193.6,'V':140.0}
_MW = {'A':89.09,'R':174.20,'N':132.12,'D':133.10,'C':121.16,'Q':146.15,'E':147.13,'G':75.03,
       'H':155.16,'I':131.17,'L':131.17,'K':146.19,'M':149.21,'F':165.19,'P':115.13,'S':105.09,
       'T':119.12,'W':204.23,'Y':181.19,'V':117.15}
_POLAR = {'A':0.0,'R':52.0,'N':3.38,'D':49.7,'C':1.48,'Q':3.53,'E':49.9,'G':0.0,
           'H':51.6,'I':0.13,'L':0.13,'K':49.5,'M':1.43,'F':0.35,'P':1.58,'S':1.67,
           'T':1.66,'W':2.10,'Y':1.61,'V':0.13}
_CHARGE = {'R':'+','K':'+','H':'+','D':'-','E':'-',
           'A':'0','N':'0','C':'0','Q':'0','G':'0','I':'0','L':'0',
           'M':'0','F':'0','P':'0','S':'0','T':'0','W':'0','Y':'0','V':'0'}

def detect_freq_cols(train_df):
    al_cols = [c for c in train_df.columns if c.startswith("AL_")]
    freq_cols = []
    for c in al_cols:
        vals = train_df[c].dropna()
        if len(vals) == 0: continue
        uniq = set(vals.unique())
        if len(uniq) > 2 and vals.min() >= 0 and vals.mean() < 0.1:
            freq_cols.append(c)
    print(f"Detected: {len(freq_cols)} freq cols from training data")
    return freq_cols

def add_fe_cftr(df, freq_cols=None):
    out = df.copy()
    a1 = out["AA_1"].astype("object")
    a2 = out["AA_2"].astype("object")

    # === GRUP E: AA substitusyon skorlari (4 feature) ===
    out["fe_grantham"] = out.apply(
        lambda r: grantham(r["AA_1"], r["AA_2"])
        if (isinstance(r["AA_1"], str) and isinstance(r["AA_2"], str)
            and r["AA_1"] in STANDARD_AA and r["AA_2"] in STANDARD_AA) else -1,
        axis=1).astype(float)
    out["fe_blosum62"] = out.apply(
        lambda r: blosum62(r["AA_1"], r["AA_2"])
        if (isinstance(r["AA_1"], str) and isinstance(r["AA_2"], str)
            and r["AA_1"] in STANDARD_AA and r["AA_2"] in STANDARD_AA) else 0,
        axis=1).astype(float)

    def _grantham_cat(val):
        if val < 0 or np.isnan(val): return np.nan
        if val == 0: return 0
        if val <= 60: return 1
        if val <= 80: return 2
        if val <= 100: return 3
        return 4
    out["fe_grantham_cat"] = out["fe_grantham"].apply(_grantham_cat)

    def _charge_change(row):
        x, y = row["AA_1"], row["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in _CHARGE and y in _CHARGE:
            cx, cy = _CHARGE[x], _CHARGE[y]
            if cx == cy: return 0
            if cx in ('+', '-') or cy in ('+', '-'): return 1
            return 0
        return np.nan
    out["fe_charge_change"] = out.apply(_charge_change, axis=1)

    # === GRUP A: FCS-tarzi Frekans x Konservasyon (3 feature) ===
    if freq_cols and len(freq_cols) > 0:
        max_pop_freq = out[freq_cols].max(axis=1)
        log_max_freq = np.log10(max_pop_freq.clip(lower=1e-10))
        log_max_freq = log_max_freq.where(max_pop_freq.notna(), np.nan)
    else:
        log_max_freq = pd.Series(np.nan, index=out.index)

    ek9 = out["EK_9"] if "EK_9" in out.columns else pd.Series(np.nan, index=out.index)
    ek7 = out["EK_7"] if "EK_7" in out.columns else pd.Series(np.nan, index=out.index)

    out["fe_fcs_ek9"] = log_max_freq * ek9
    out["fe_fcs_ek7"] = log_max_freq * ek7
    out["fe_log_max_freq"] = log_max_freq

    # === GRUP B: EK Skor Birlesimleri (5 feature) ===
    ek_map = {}
    for i in range(1, 10):
        cn = f"EK_{i}"
        if cn in out.columns:
            ek_map[i] = out[cn]

    if all(k in ek_map for k in [7, 9, 2]):
        out["fe_ek_mean_top3"] = pd.concat([ek_map[7], ek_map[9], ek_map[2]], axis=1).mean(axis=1, skipna=True)
    else:
        out["fe_ek_mean_top3"] = np.nan

    if all(k in ek_map for k in [7, 9, 2, 6]):
        four = pd.concat([ek_map[7], ek_map[9], ek_map[2], ek_map[6]], axis=1)
        out["fe_ek_max"] = four.max(axis=1, skipna=True)
        out["fe_ek_range"] = four.max(axis=1, skipna=True) - four.min(axis=1, skipna=True)
    else:
        out["fe_ek_max"] = np.nan
        out["fe_ek_range"] = np.nan

    if all(k in ek_map for k in [1, 2]):
        out["fe_ek_delta_12"] = ek_map[1] - ek_map[2]
    else:
        out["fe_ek_delta_12"] = np.nan

    if all(k in ek_map for k in [4, 5, 6]):
        out["fe_ek_consensus"] = ek_map[4] + ek_map[5] + ek_map[6]
    else:
        out["fe_ek_consensus"] = np.nan

    # === GRUP D: AA Fizikokimyasal Delta (4 feature) ===
    def _aa_delta_abs(lookup):
        def _calc(row):
            x, y = row["AA_1"], row["AA_2"]
            if isinstance(x, str) and isinstance(y, str) and x in lookup and y in lookup:
                return abs(lookup[x] - lookup[y])
            return np.nan
        return out.apply(_calc, axis=1)

    out["fe_hydro_abs"] = _aa_delta_abs(_HYDRO)
    out["fe_vol_abs"]   = _aa_delta_abs(_VOLUME)
    out["fe_mw_abs"]    = _aa_delta_abs(_MW)
    out["fe_polar_abs"] = _aa_delta_abs(_POLAR)

    return out

# Feature group definitions (CFTR: no Group C)
FE_GROUP_A = ["fe_fcs_ek9", "fe_fcs_ek7", "fe_log_max_freq"]
FE_GROUP_B = ["fe_ek_mean_top3", "fe_ek_max", "fe_ek_range", "fe_ek_delta_12", "fe_ek_consensus"]
FE_GROUP_D = ["fe_hydro_abs", "fe_vol_abs", "fe_mw_abs", "fe_polar_abs"]
FE_GROUP_E = ["fe_grantham", "fe_blosum62", "fe_grantham_cat", "fe_charge_change"]
FE_ALL_CFTR = FE_GROUP_A + FE_GROUP_B + FE_GROUP_D + FE_GROUP_E

print(f"add_fe_cftr hazir. Gruplar: A={len(FE_GROUP_A)}, B={len(FE_GROUP_B)}, "
      f"D={len(FE_GROUP_D)}, E={len(FE_GROUP_E)}, Toplam={len(FE_ALL_CFTR)}")
print(f"Feature listesi: {FE_ALL_CFTR}")

add_fe_cftr hazir. Gruplar: A=3, B=5, D=4, E=4, Toplam=16
Feature listesi: ['fe_fcs_ek9', 'fe_fcs_ek7', 'fe_log_max_freq', 'fe_ek_mean_top3', 'fe_ek_max', 'fe_ek_range', 'fe_ek_delta_12', 'fe_ek_consensus', 'fe_hydro_abs', 'fe_vol_abs', 'fe_mw_abs', 'fe_polar_abs', 'fe_grantham', 'fe_blosum62', 'fe_grantham_cat', 'fe_charge_change']


In [4]:
# Cell 4: M3 Preprocessing + Degerlendirme Altyapisi

LGBM_FIXED = {
    'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
    'verbosity': -1, 'random_state': SEED, 'objective': 'binary',
    'class_weight': 'balanced',
}

def fit_preprocessor(train_df_raw, fe_cols=None, freq_cols_list=None):
    if fe_cols:
        tr = add_fe_cftr(train_df_raw, freq_cols=freq_cols_list)
    else:
        tr = train_df_raw.copy()
    fe_num = [c for c in (fe_cols or []) if c in tr.columns]
    num_cols = NUM_COLS_BASE + fe_num
    miss = tr[base_feature_cols].isna().mean()
    flag_source = miss[miss > HIGH_MISSING_THR].index.tolist()
    median = {c: pd.to_numeric(tr[c], errors="coerce").median() for c in num_cols}
    return {"num_cols": num_cols, "cat_cols": CAT_LIKE,
            "flag_source": flag_source, "median": median,
            "fe_cols": fe_cols or [], "freq_cols_list": freq_cols_list}

def transform_X(df_raw, pp):
    if pp["fe_cols"]:
        df = add_fe_cftr(df_raw, freq_cols=pp["freq_cols_list"])
    else:
        df = df_raw.copy()
    out = pd.DataFrame(index=df.index)
    for c in pp["num_cols"]:
        out[c] = pd.to_numeric(df[c], errors="coerce").fillna(pp["median"].get(c, 0)).astype(float).values
    for c in pp["cat_cols"]:
        fill = AA_UNK if c in CR.AA_COLS else "MISSING"
        out[c] = df[c].astype("object").where(~df[c].isna(), fill).astype(str).values
    for c in pp["flag_source"]:
        out[CR.get_missing_mask_col_name(c)] = df_raw[c].isna().astype(int).values
    return out

def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y); prob = np.asarray(prob)
    neg = np.where(y == 0)[0]; pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0: return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED); f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    return {"mean": float(np.mean(f1s)), "std": float(np.std(f1s)),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_robust(y, prob, n=50):
    rng = np.random.RandomState(BOOT_SEED)
    thresholds = []
    for _ in range(n):
        yb, pb = _resample_8020(y, prob, rng)
        best, best_thr = -1.0, 0.5
        for thr in np.arange(0.05, 0.95, 0.01):
            f = _f1_pos(yb, (pb >= thr).astype(int))
            if f > best: best, best_thr = f, thr
        thresholds.append(best_thr)
    return float(np.mean(thresholds))

def _le_encode(X_df, cat_cols):
    Xn = X_df.copy()
    for c in cat_cols:
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c].astype(str))
    return Xn.astype(float)

def _lgbm_classifier():
    return lgb.LGBMClassifier(**{**LGBM_FIXED, "n_estimators": 200,
                                  "num_leaves": 31, "learning_rate": 0.05})

def run_cftr_eval(exp_name, combined_df, cftr_df, fe_cols=None, freq_cols_list=None):
    pp = fit_preprocessor(combined_df, fe_cols=fe_cols, freq_cols_list=freq_cols_list)
    X_combined = transform_X(combined_df, pp)
    y_combined = combined_df[TARGET].values
    X_cftr = transform_X(cftr_df, pp)
    y_cftr = cftr_df[TARGET].values
    cat_cols = list(pp["cat_cols"])

    X_comb_le = _le_encode(X_combined, cat_cols)
    X_cftr_le = _le_encode(X_cftr, cat_cols)
    pi_train = float(y_combined.mean())

    m = _lgbm_classifier()
    m.fit(X_comb_le, y_combined)
    p_raw = m.predict_proba(X_cftr_le)[:, 1]
    p_adj = adjust_prior_shift(p_raw, pi_train)

    thr = select_threshold_robust(y_cftr, p_adj)
    y_pred = (p_adj >= thr).astype(int)
    mcc = matthews_corrcoef(y_cftr, y_pred)
    f1 = _f1_pos(y_cftr, y_pred)
    prec = precision_score(y_cftr, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_cftr, y_pred, pos_label=1, zero_division=0)
    auc = roc_auc_score(y_cftr, p_adj) if len(np.unique(y_cftr)) > 1 else 0.0
    boot = bootstrap_8020(y_cftr, p_adj, thr)
    cm = confusion_matrix(y_cftr, y_pred, labels=[0, 1])
    fi = pd.Series(m.feature_importances_, index=X_comb_le.columns)

    return {
        "experiment": exp_name, "loo_mcc": round(mcc, 4),
        "f1": round(f1, 4), "precision": round(prec, 4), "recall": round(rec, 4),
        "auc": round(auc, 4), "thr": round(thr, 4),
        "boot_mean": round(boot["mean"], 4), "boot_std": round(boot["std"], 4),
        "boot_lo": round(boot["lo"], 4), "boot_hi": round(boot["hi"], 4),
        "TN": int(cm[0,0]), "FP": int(cm[0,1]), "FN": int(cm[1,0]), "TP": int(cm[1,1]),
        "n_features": X_comb_le.shape[1],
        "fi": fi, "y_pred": y_pred, "p_adj": p_adj
    }

print("Preprocessing + Degerlendirme altyapisi hazir.")

Preprocessing + Degerlendirme altyapisi hazir.


In [5]:
# Cell 5: Frekans sutunlarini COMBINED uzerinde tespit et (leakage-free)
freq_cols = detect_freq_cols(combined)
print(f"Frekans sutun sayisi: {len(freq_cols)}")
print(f"Ilk 10: {freq_cols[:10]}")

Detected: 119 freq cols from training data
Frekans sutun sayisi: 119
Ilk 10: ['AL_1', 'AL_2', 'AL_3', 'AL_4', 'AL_5', 'AL_6', 'AL_7', 'AL_8', 'AL_9', 'AL_10']


In [6]:
# Cell 6: Exp 1 -- Grup Ablasyonu
print("="*70)
print("[Exp 1] CFTR Feature Group Ablation")
print("="*70)

ablation_configs = {
    "E1a_GroupA_FCS":        FE_GROUP_A,
    "E1b_GroupB_EKCombo":    FE_GROUP_B,
    "E1c_GroupD_AAphyschem": FE_GROUP_D,
    "E1d_GroupE_AAsubst":    FE_GROUP_E,
    "E1e_All_ABDE":          FE_ALL_CFTR,
    "E1f_NoFE":              None,
}

all_results = {}

for exp_name, fe_cols in ablation_configs.items():
    print(f"\n--- {exp_name} ({len(fe_cols) if fe_cols else 0} FE features) ---")
    try:
        res = run_cftr_eval(exp_name, combined, cftr,
                            fe_cols=fe_cols, freq_cols_list=freq_cols)
        all_results[exp_name] = res
        print(f"  LOO-MCC={res['loo_mcc']:.4f}  Boot-F1={res['boot_mean']:.4f} +/- {res['boot_std']:.3f}  "
              f"Prec={res['precision']:.3f}  TN={res['TN']}  FP={res['FP']}  FN={res['FN']}  TP={res['TP']}")
    except Exception as e:
        print(f"  HATA: {e}")
        import traceback; traceback.print_exc()
        all_results[exp_name] = None

# Summary table
print("\n" + "="*70)
print("EXP 1 OZET:")
print(f"{'Deney':<25} {'LOO-MCC':>8} {'Boot-F1':>8} {'Boot-std':>9} {'Prec':>6} {'TN':>4} {'FP':>4} {'FN':>4} {'TP':>4}")
print("-"*80)
baseline_mcc = 0.6436
for name, res in all_results.items():
    if res is None: continue
    delta = res["loo_mcc"] - baseline_mcc
    print(f"{name:<25} {res['loo_mcc']:>8.4f} {res['boot_mean']:>8.4f} {res['boot_std']:>9.4f} "
          f"{res['precision']:>6.3f} {res['TN']:>4} {res['FP']:>4} {res['FN']:>4} {res['TP']:>4}  "
          f"({'+'if delta>=0 else ''}{delta:.4f})")
print("="*70)
print(f"NB20 Baseline: LOO-MCC=0.6436, Boot-F1=0.8629")

[Exp 1] CFTR Feature Group Ablation

--- E1a_GroupA_FCS (3 FE features) ---
  LOO-MCC=0.4913  Boot-F1=0.6423 +/- 0.111  Prec=0.970  TN=19  FP=2  FN=26  TP=64

--- E1b_GroupB_EKCombo (5 FE features) ---
  LOO-MCC=0.4808  Boot-F1=0.6343 +/- 0.129  Prec=0.969  TN=19  FP=2  FN=27  TP=63

--- E1c_GroupD_AAphyschem (4 FE features) ---
  LOO-MCC=0.5381  Boot-F1=0.7139 +/- 0.126  Prec=0.985  TN=20  FP=1  FN=25  TP=65

--- E1d_GroupE_AAsubst (4 FE features) ---
  LOO-MCC=0.5602  Boot-F1=0.7240 +/- 0.134  Prec=0.985  TN=20  FP=1  FN=23  TP=67

--- E1e_All_ABDE (16 FE features) ---
  LOO-MCC=0.6074  Boot-F1=0.8367 +/- 0.114  Prec=1.000  TN=21  FP=0  FN=22  TP=68

--- E1f_NoFE (0 FE features) ---
  LOO-MCC=0.5243  Boot-F1=0.6754 +/- 0.127  Prec=0.971  TN=19  FP=2  FN=23  TP=67

EXP 1 OZET:
Deney                      LOO-MCC  Boot-F1  Boot-std   Prec   TN   FP   FN   TP
--------------------------------------------------------------------------------
E1a_GroupA_FCS              0.4913   0.6423    0.

In [7]:
# Cell 7: Exp 2 -- FE Versiyon Karsilastirmasi
print("\n" + "="*70)
print("[Exp 2] FE Versiyon Karsilastirmasi")
print("="*70)

FE_NB16_CFTR = ["fe_grantham", "fe_blosum62"]

# En iyi 2 grup: Exp 1 sonuclarina gore
e1_scored = [(k, v["loo_mcc"]) for k, v in all_results.items()
             if v and k not in ("E1f_NoFE", "E1e_All_ABDE")]
e1_scored.sort(key=lambda x: x[1], reverse=True)
group_map = {
    "E1a_GroupA_FCS": FE_GROUP_A,
    "E1b_GroupB_EKCombo": FE_GROUP_B,
    "E1c_GroupD_AAphyschem": FE_GROUP_D,
    "E1d_GroupE_AAsubst": FE_GROUP_E,
}
if len(e1_scored) >= 2:
    best2_names = [e1_scored[0][0], e1_scored[1][0]]
    best2_cols = []
    for n in best2_names:
        best2_cols.extend(group_map.get(n, []))
    print(f"En iyi 2 grup: {best2_names} -> {len(best2_cols)} feature")
else:
    best2_cols = FE_ALL_CFTR

exp2_configs = {
    "E2a_NoFE":         None,
    "E2b_NB16_FE":      FE_NB16_CFTR,
    "E2c_Full_CFTR_FE": FE_ALL_CFTR,
    "E2d_Best2_Groups": best2_cols if best2_cols else FE_ALL_CFTR,
}

for exp_name, fe_cols in exp2_configs.items():
    print(f"\n--- {exp_name} ({len(fe_cols) if fe_cols else 0} FE features) ---")
    try:
        res = run_cftr_eval(exp_name, combined, cftr,
                            fe_cols=fe_cols, freq_cols_list=freq_cols)
        all_results[exp_name] = res
        print(f"  LOO-MCC={res['loo_mcc']:.4f}  Boot-F1={res['boot_mean']:.4f} +/- {res['boot_std']:.3f}  "
              f"Prec={res['precision']:.3f}  TN={res['TN']}  FP={res['FP']}  FN={res['FN']}  TP={res['TP']}")
    except Exception as e:
        print(f"  HATA: {e}")
        import traceback; traceback.print_exc()
        all_results[exp_name] = None

print("\n" + "="*70)
print("EXP 2 OZET:")
print(f"{'Deney':<25} {'LOO-MCC':>8} {'Boot-F1':>8} {'Prec':>6} {'FP':>4} {'FN':>4} {'n_feat':>7}")
print("-"*65)
for name, res in all_results.items():
    if res is None or not name.startswith("E2"): continue
    print(f"{name:<25} {res['loo_mcc']:>8.4f} {res['boot_mean']:>8.4f} "
          f"{res['precision']:>6.3f} {res['FP']:>4} {res['FN']:>4} {res['n_features']:>7}")
print("="*70)


[Exp 2] FE Versiyon Karsilastirmasi
En iyi 2 grup: ['E1d_GroupE_AAsubst', 'E1c_GroupD_AAphyschem'] -> 8 feature

--- E2a_NoFE (0 FE features) ---
  LOO-MCC=0.5243  Boot-F1=0.6754 +/- 0.127  Prec=0.971  TN=19  FP=2  FN=23  TP=67

--- E2b_NB16_FE (2 FE features) ---
  LOO-MCC=0.6081  Boot-F1=0.7644 +/- 0.120  Prec=0.986  TN=20  FP=1  FN=19  TP=71

--- E2c_Full_CFTR_FE (16 FE features) ---
  LOO-MCC=0.6074  Boot-F1=0.8367 +/- 0.114  Prec=1.000  TN=21  FP=0  FN=22  TP=68

--- E2d_Best2_Groups (8 FE features) ---
  LOO-MCC=0.5020  Boot-F1=0.6438 +/- 0.117  Prec=0.970  TN=19  FP=2  FN=25  TP=65

EXP 2 OZET:
Deney                      LOO-MCC  Boot-F1   Prec   FP   FN  n_feat
-----------------------------------------------------------------
E2a_NoFE                    0.5243   0.6754  0.971    2   23     428
E2b_NB16_FE                 0.6081   0.7644  0.986    1   19     430
E2c_Full_CFTR_FE            0.6074   0.8367  1.000    0   22     444
E2d_Best2_Groups            0.5020   0.6438  0.9

In [8]:
# Cell 8: Feature Importance + Korelasyon Dogrulamasi
print("\n" + "="*70)
print("Feature Importance + Korelasyon Analizi")
print("="*70)

# En iyi FE modeli
best_key = max(
    [(k, v["loo_mcc"]) for k, v in all_results.items() if v and k not in ("E1f_NoFE", "E2a_NoFE")],
    key=lambda x: x[1]
)[0]
best_res = all_results[best_key]
fi = best_res["fi"]

fe_fi = fi[[c for c in FE_ALL_CFTR if c in fi.index]].sort_values(ascending=False)
print(f"\nEn iyi model: {best_key} (LOO-MCC={best_res['loo_mcc']:.4f})")
print(f"\nFE Feature Importance (top):")
for feat, imp in fe_fi.items():
    total_pct = imp / fi.sum() * 100
    print(f"  {feat:<25} {imp:>10.1f}  ({total_pct:>5.2f}%)")

print(f"\nTop-20 Overall Feature Importance:")
top20 = fi.sort_values(ascending=False).head(20)
for feat, imp in top20.items():
    marker = " <-- FE" if feat in FE_ALL_CFTR else ""
    print(f"  {feat:<25} {imp:>10.1f}{marker}")

fi_df = pd.DataFrame({"feature": fi.index, "importance": fi.values})
fi_df = fi_df.sort_values("importance", ascending=False).reset_index(drop=True)
fi_df.to_csv(os.path.join(RESULTS_DIR, "feature_importance.csv"), index=False)

# Korelasyon dogrulamasi
print("\n" + "-"*70)
print("CFTR FE Korelasyon Dogrulamasi")
cftr_fe = add_fe_cftr(cftr, freq_cols=freq_cols)
y_cftr_arr = cftr[TARGET].values

print(f"\n{'Feature':<25} {'r(Label)':>10} {'n_valid':>8} {'Kategori':<20}")
print("-"*70)
for feat in FE_ALL_CFTR:
    if feat not in cftr_fe.columns: continue
    vals = pd.to_numeric(cftr_fe[feat], errors="coerce")
    mask = vals.notna()
    if mask.sum() < 10:
        print(f"  {feat:<25} {'N/A':>10} {mask.sum():>8}")
        continue
    r = np.corrcoef(vals[mask].values, y_cftr_arr[mask])[0, 1]
    if feat in FE_GROUP_A: cat = "Grup A (FCS)"
    elif feat in FE_GROUP_B: cat = "Grup B (EK combo)"
    elif feat in FE_GROUP_D: cat = "Grup D (AA phys)"
    elif feat in FE_GROUP_E: cat = "Grup E (AA subst)"
    else: cat = "?"
    print(f"  {feat:<25} {r:>10.4f} {mask.sum():>8} {cat:<20}")


Feature Importance + Korelasyon Analizi

En iyi model: E2b_NB16_FE (LOO-MCC=0.6081)

FE Feature Importance (top):
  fe_grantham                    116.0  ( 1.93%)
  fe_blosum62                     89.0  ( 1.48%)

Top-20 Overall Feature Importance:
  EK_9                           179.0
  EK_7                           164.0
  AA_2                           156.0
  EK_2                           150.0
  EK_8                           131.0
  EK_1                           126.0
  CAT_1                          122.0
  AL_26                          119.0
  EK_5                           118.0
  fe_grantham                    116.0 <-- FE
  AA_1                           107.0
  AL_12                          107.0
  EK_3                           105.0
  EK_4                            98.0
  AL_13                           89.0
  fe_blosum62                     89.0 <-- FE
  AL_20                           86.0
  AL_8                            82.0
  EK_6                            8

In [9]:
# Cell 9: Gorsellestirmeler + CSV

# Tum sonuclari topla
rows = []
for k, v in sorted(all_results.items()):
    if v is None: continue
    rows.append({k2: v2 for k2, v2 in v.items() if k2 not in ("fi", "y_pred", "p_adj")})
results_df = pd.DataFrame(rows)
csv_path = os.path.join(RESULTS_DIR, "cftr_fe_results.csv")
results_df.to_csv(csv_path, index=False)
print(f"Sonuclar: {csv_path}")
print(results_df[["experiment","loo_mcc","boot_mean","boot_std","precision","FP","FN","n_features"]].to_string(index=False))

# Fig 1: Group Ablation
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig1.suptitle("NB34 - CFTR Feature Engineering Ablation", fontweight="bold", fontsize=13)

e1_names = list(ablation_configs.keys())
e1_mcc = [all_results.get(k, {}).get("loo_mcc", 0) if all_results.get(k) else 0 for k in e1_names]
e1_boot = [all_results.get(k, {}).get("boot_mean", 0) if all_results.get(k) else 0 for k in e1_names]
colors = ['#2196F3' if k != "E1f_NoFE" else '#FF5722' for k in e1_names]
short_names = [n.split("_", 1)[1] for n in e1_names]

bars = ax1.bar(range(len(e1_names)), e1_mcc, color=colors, edgecolor="black", linewidth=0.8)
ax1.axhline(y=0.6436, color='red', linestyle='--', linewidth=1.5, label='NB20 Baseline (0.644)')
for bar, val in zip(bars, e1_mcc):
    if val > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)
ax1.set_xticks(range(len(e1_names)))
ax1.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
ax1.set_ylabel("LOO-CV MCC"); ax1.set_title("Exp 1: Grup Ablasyonu (LOO-MCC)")
ax1.set_ylim(0, 0.85); ax1.legend()

bars2 = ax2.bar(range(len(e1_names)), e1_boot, color=colors, edgecolor="black", linewidth=0.8)
ax2.axhline(y=0.8629, color='red', linestyle='--', linewidth=1.5, label='NB20 Baseline (0.863)')
for bar, val in zip(bars2, e1_boot):
    if val > 0:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)
ax2.set_xticks(range(len(e1_names)))
ax2.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel("Bootstrap 80/20 F1"); ax2.set_title("Exp 1: Grup Ablasyonu (Boot-F1)")
ax2.set_ylim(0, 1.05); ax2.legend()

plt.tight_layout()
fig1.savefig(os.path.join(RESULTS_DIR, "fig1_group_ablation.png"), dpi=150, bbox_inches="tight")
plt.close(fig1)
print("fig1_group_ablation.png kaydedildi.")

# Fig 2: FE Version Comparison
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(12, 5))
fig2.suptitle("NB34 - CFTR FE Version Comparison", fontweight="bold", fontsize=13)

e2_names = list(exp2_configs.keys())
e2_mcc = [all_results.get(k, {}).get("loo_mcc", 0) if all_results.get(k) else 0 for k in e2_names]
e2_boot = [all_results.get(k, {}).get("boot_mean", 0) if all_results.get(k) else 0 for k in e2_names]
e2_short = [n.split("_", 1)[1] for n in e2_names]
e2_colors = ['#FF5722', '#FFC107', '#4CAF50', '#2196F3']

bars3 = ax3.bar(range(len(e2_names)), e2_mcc, color=e2_colors, edgecolor="black", linewidth=0.8)
ax3.axhline(y=0.6436, color='red', linestyle='--', linewidth=1.5, label='NB20 Baseline')
for bar, val in zip(bars3, e2_mcc):
    if val > 0:
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
ax3.set_xticks(range(len(e2_names)))
ax3.set_xticklabels(e2_short, rotation=30, ha='right', fontsize=9)
ax3.set_ylabel("LOO-CV MCC"); ax3.set_title("Exp 2: LOO-MCC")
ax3.set_ylim(0, 0.85); ax3.legend()

bars4 = ax4.bar(range(len(e2_names)), e2_boot, color=e2_colors, edgecolor="black", linewidth=0.8)
ax4.axhline(y=0.8629, color='red', linestyle='--', linewidth=1.5, label='NB20 Baseline')
for bar, val in zip(bars4, e2_boot):
    if val > 0:
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
ax4.set_xticks(range(len(e2_names)))
ax4.set_xticklabels(e2_short, rotation=30, ha='right', fontsize=9)
ax4.set_ylabel("Bootstrap 80/20 F1"); ax4.set_title("Exp 2: Boot-F1")
ax4.set_ylim(0, 1.05); ax4.legend()

plt.tight_layout()
fig2.savefig(os.path.join(RESULTS_DIR, "fig2_fe_comparison.png"), dpi=150, bbox_inches="tight")
plt.close(fig2)
print("fig2_fe_comparison.png kaydedildi.")

# Fig 3: Feature Importance (FE only)
fi = best_res["fi"]
fe_fi = fi[[c for c in FE_ALL_CFTR if c in fi.index]].sort_values(ascending=True)
if len(fe_fi) > 0:
    fig3, ax5 = plt.subplots(figsize=(8, max(4, len(fe_fi)*0.35)))
    colors_fi = []
    for feat in fe_fi.index:
        if feat in FE_GROUP_A: colors_fi.append('#2196F3')
        elif feat in FE_GROUP_B: colors_fi.append('#4CAF50')
        elif feat in FE_GROUP_D: colors_fi.append('#FF9800')
        elif feat in FE_GROUP_E: colors_fi.append('#9C27B0')
        else: colors_fi.append('#607D8B')
    ax5.barh(range(len(fe_fi)), fe_fi.values, color=colors_fi, edgecolor="black", linewidth=0.5)
    ax5.set_yticks(range(len(fe_fi)))
    ax5.set_yticklabels(fe_fi.index, fontsize=8)
    ax5.set_xlabel("Feature Importance (gain)")
    ax5.set_title(f"NB34 - FE Feature Importance ({best_key})")
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#2196F3', label='A: FCS'),
                      Patch(facecolor='#4CAF50', label='B: EK combo'),
                      Patch(facecolor='#FF9800', label='D: AA phys'),
                      Patch(facecolor='#9C27B0', label='E: AA subst')]
    ax5.legend(handles=legend_elements, loc='lower right', fontsize=8)
    plt.tight_layout()
    fig3.savefig(os.path.join(RESULTS_DIR, "fig3_feature_importance.png"), dpi=150, bbox_inches="tight")
    plt.close(fig3)
    print("fig3_feature_importance.png kaydedildi.")

# Fig 4: Confusion matrix
best_overall = max([(k, v) for k, v in all_results.items() if v], key=lambda x: x[1]["loo_mcc"])
best_name, best_r = best_overall
cm = np.array([[best_r["TN"], best_r["FP"]], [best_r["FN"], best_r["TP"]]])
fig4, ax6 = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=["Benign", "Pathogenic"])
disp.plot(ax=ax6, cmap="Blues", values_format="d")
ax6.set_title(f"Best: {best_name}\nLOO-MCC={best_r['loo_mcc']:.4f}, Boot-F1={best_r['boot_mean']:.4f}")
plt.tight_layout()
fig4.savefig(os.path.join(RESULTS_DIR, "fig4_confusion_best.png"), dpi=150, bbox_inches="tight")
plt.close(fig4)
print("fig4_confusion_best.png kaydedildi.")

Sonuclar: /Users/tefe/teknofest_model/teknofest_model/results/v12_cftr_fe/cftr_fe_results.csv
           experiment  loo_mcc  boot_mean  boot_std  precision  FP  FN  n_features
       E1a_GroupA_FCS   0.4913     0.6423    0.1106     0.9697   2  26         431
   E1b_GroupB_EKCombo   0.4808     0.6343    0.1286     0.9692   2  27         433
E1c_GroupD_AAphyschem   0.5381     0.7139    0.1256     0.9848   1  25         432
   E1d_GroupE_AAsubst   0.5602     0.7240    0.1339     0.9853   1  23         432
         E1e_All_ABDE   0.6074     0.8367    0.1136     1.0000   0  22         444
             E1f_NoFE   0.5243     0.6754    0.1269     0.9710   2  23         428
             E2a_NoFE   0.5243     0.6754    0.1269     0.9710   2  23         428
          E2b_NB16_FE   0.6081     0.7644    0.1198     0.9861   1  19         430
     E2c_Full_CFTR_FE   0.6074     0.8367    0.1136     1.0000   0  22         444
     E2d_Best2_Groups   0.5020     0.6438    0.1169     0.9701   2  25      

In [10]:
# Cell 10: PDF Rapor
from fpdf import FPDF

class NB34Report(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.cell(0, 6, "NB34 - CFTR Feature Engineering Ablation", align="C", new_x="LMARGIN", new_y="NEXT")
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(3)
    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Sayfa {self.page_no()}/{{nb}}", align="C")
    def section(self, title):
        self.set_font("Helvetica", "B", 12)
        self.cell(0, 8, title, new_x="LMARGIN", new_y="NEXT")
        self.ln(2)
    def body_text(self, txt):
        self.set_font("Helvetica", "", 9)
        self.multi_cell(0, 5, txt)
        self.ln(2)
    def add_table(self, headers, rows, col_widths=None):
        if col_widths is None:
            col_widths = [190 // len(headers)] * len(headers)
        self.set_font("Helvetica", "B", 8)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, str(h), border=1, align="C")
        self.ln()
        self.set_font("Helvetica", "", 7)
        for row in rows:
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), border=1, align="C")
            self.ln()
        self.ln(3)

pdf = NB34Report()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=20)

# Page 1
pdf.add_page()
pdf.set_font("Helvetica", "B", 16)
pdf.cell(0, 15, "NB34: CFTR Feature Engineering", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "", 10)
pdf.cell(0, 8, f"Tarih: {datetime.now().strftime('%Y-%m-%d %H:%M')}", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.ln(10)

pdf.section("1. Motivasyon")
pdf.body_text(
    "NB20 S0c_COMBINED modeli FE olmadan LOO-MCC=0.644, Boot-F1=0.863, Precision=1.0 elde ediyor.\n"
    "Literatur arastirmasi (FCS, CFTR-MetaPred, AlphaMissense, PHACTboost) sonucu\n"
    "16 feature (5 grup) belirlendi. Missingness (Grup C) CFTR'de anlamsiz (r<0.05).\n"
    "Hedef: LOO-MCC = 0.644 -> 0.67-0.70 arasi."
)

pdf.section("2. Feature Gruplari")
pdf.body_text(
    "Grup A (3): FCS-tarzi frekans x konservasyon (fcs_ek9 r=-0.492)\n"
    "Grup B (5): EK skor birlesimleri (ek_mean_top3 r=0.476)\n"
    "Grup D (4): AA fizikokimyasal delta (hydro_abs r=0.170)\n"
    "Grup E (4): AA substitusyon (grantham, blosum62, grantham_cat, charge_change)\n"
    "Grup C: CIKARILDI -- tek-gen panelinde missingness homojen"
)

# Results table
pdf.section("3. Sonuclar")
headers = ["Deney", "LOO-MCC", "Boot-F1", "Std", "Prec", "FP", "FN"]
col_w = [45, 22, 22, 18, 18, 12, 12]
table_rows = []
for k, v in sorted(all_results.items()):
    if v is None: continue
    table_rows.append([
        v["experiment"], f"{v['loo_mcc']:.4f}", f"{v['boot_mean']:.4f}",
        f"{v['boot_std']:.4f}", f"{v['precision']:.3f}", str(v["FP"]), str(v["FN"])
    ])
pdf.add_table(headers, table_rows, col_w)

best_all = max([(k,v) for k,v in all_results.items() if v], key=lambda x: x[1]["loo_mcc"])
delta = best_all[1]["loo_mcc"] - 0.6436
pdf.body_text(
    f"En iyi model: {best_all[0]}\n"
    f"LOO-MCC: {best_all[1]['loo_mcc']:.4f} (delta={'+' if delta>=0 else ''}{delta:.4f} vs NB20)\n"
    f"Boot-F1: {best_all[1]['boot_mean']:.4f} +/- {best_all[1]['boot_std']:.4f}\n"
    f"CM: TN={best_all[1]['TN']}, FP={best_all[1]['FP']}, FN={best_all[1]['FN']}, TP={best_all[1]['TP']}"
)

# Figures
pdf.section("4. Gorseller")
for fig_name in ["fig1_group_ablation.png", "fig2_fe_comparison.png",
                  "fig3_feature_importance.png", "fig4_confusion_best.png"]:
    fig_path = os.path.join(RESULTS_DIR, fig_name)
    if os.path.exists(fig_path):
        pdf.image(fig_path, w=170)
        pdf.ln(3)

# Conclusion
pdf.section("5. Yorum ve Karar")
if delta > 0.05:
    verdict = f"FE CFTR'de anlamli iyilestirme sagladi (+{delta:.4f}). En iyi FE konfigurasyonu sabitleniyor."
elif delta > 0:
    verdict = f"FE kucuk iyilestirme (+{delta:.4f}), guerueltue bandinda (< 0.05). Ek dogrulama gerekir."
else:
    verdict = f"FE CFTR'de deger katmadi ({delta:+.4f}). NB20 baseline korunuyor."

pdf.body_text(
    f"{verdict}\n\n"
    "CFTR'de n=21 benign nedeniyle MCC farki > 0.05 olmadan 'kazanan' ilan edilmez.\n"
    "Boot-CI=[0.00-1.00] nedeniyle Boot-F1 TEK BASINA karar kriteri DEGIL."
)

pdf_path = os.path.join(REPORTS_DIR, "NB34_cftr_fe_report.pdf")
pdf.output(pdf_path)
print(f"PDF rapor kaydedildi: {pdf_path}")

PDF rapor kaydedildi: /Users/tefe/teknofest_model/teknofest_model/reports/NB34_cftr_fe_report.pdf


In [11]:
# Cell 11: Ozet
print("\n" + "="*70)
print("NB34 TAMAMLANDI")
print("="*70)

best_all = max([(k,v) for k,v in all_results.items() if v], key=lambda x: x[1]["loo_mcc"])
baseline_mcc = 0.6436
delta = best_all[1]["loo_mcc"] - baseline_mcc

print(f"\nBaseline (NB20): LOO-MCC=0.6436, Boot-F1=0.8629")
print(f"En iyi FE    : {best_all[0]}")
print(f"  LOO-MCC    : {best_all[1]['loo_mcc']:.4f} (delta={'+' if delta>=0 else ''}{delta:.4f})")
print(f"  Boot-F1    : {best_all[1]['boot_mean']:.4f} +/- {best_all[1]['boot_std']:.4f}")
print(f"  Precision  : {best_all[1]['precision']:.4f}")
print(f"  CM         : TN={best_all[1]['TN']}, FP={best_all[1]['FP']}, FN={best_all[1]['FN']}, TP={best_all[1]['TP']}")

if delta > 0.05:
    print(f"\n>> KARAR: FE anlamli iyilestirme sagladi. En iyi FE sabitleniyor.")
elif delta > 0:
    print(f"\n>> KARAR: FE kucuk iyilestirme, guerueltue bandinda. NB20 baseline korunabilir.")
else:
    print(f"\n>> KARAR: FE deger katmadi. NB20 baseline korunuyor.")

print(f"\nCiktilar: {RESULTS_DIR}/")
print(f"  cftr_fe_results.csv")
print(f"  feature_importance.csv")
print(f"  fig1_group_ablation.png")
print(f"  fig2_fe_comparison.png")
print(f"  fig3_feature_importance.png")
print(f"  fig4_confusion_best.png")
print(f"  reports/NB34_cftr_fe_report.pdf")
print("="*70)


NB34 TAMAMLANDI

Baseline (NB20): LOO-MCC=0.6436, Boot-F1=0.8629
En iyi FE    : E2b_NB16_FE
  LOO-MCC    : 0.6081 (delta=-0.0355)
  Boot-F1    : 0.7644 +/- 0.1198
  Precision  : 0.9861
  CM         : TN=20, FP=1, FN=19, TP=71

>> KARAR: FE deger katmadi. NB20 baseline korunuyor.

Ciktilar: /Users/tefe/teknofest_model/teknofest_model/results/v12_cftr_fe/
  cftr_fe_results.csv
  feature_importance.csv
  fig1_group_ablation.png
  fig2_fe_comparison.png
  fig3_feature_importance.png
  fig4_confusion_best.png
  reports/NB34_cftr_fe_report.pdf
